In [ ]:
import gc
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import qutip

from codes.codewords import bgmcode_piqs, gross_13_piqs, gross_17_piqs, seven_qubit_piqs
from codes.noisemodel import noisemodel
from codes.optimisation import no_recovery, optimise

os.environ["MOSEKLM_LICENSE_FILE"] = "/Users/a46668993/Desktop/qer/mosek/mosek.lic"

# Sweep parameters: replace these with the values you want to study.
gamma_vals = np.logspace(-5, -2, 10)
dt = 1.0
p_vals = gamma_vals * dt

# Solver / channel settings.
method = "choi"
solver = "mosek"
noise_name = "local symmetric amplitude damping"

# Code definitions in PIQS reduced form.
rho_seven, l0_seven, l1_seven = seven_qubit_piqs(return_qutip=True)
rho_bgm, l0_bgm, l1_bgm = bgmcode_piqs(3, 3, 1, return_qutip=True)
rho_gross13, l0_gross13, l1_gross13 = gross_13_piqs(return_qutip=True)
rho_gross17, l0_gross17, l1_gross17 = gross_17_piqs(return_qutip=True)
rho_bare = qutip.qeye(2) / 2
l0_bare = qutip.basis(2, 0)
l1_bare = qutip.basis(2, 1)

codes = {
    "seven-qubit": {"N": 7, "rho": rho_seven, "l0": l0_seven, "l1": l1_seven},
    "bgm(3,3,1)": {"N": 2 * 3 * 1 + 3, "rho": rho_bgm, "l0": l0_bgm, "l1": l1_bgm},
    "gross13": {"N": 13, "rho": rho_gross13, "l0": l0_gross13, "l1": l1_gross13},
    "gross17": {"N": 17, "rho": rho_gross17, "l0": l0_gross17, "l1": l1_gross17},
    "bare qubit": {"N": 1, "rho": rho_bare, "l0": l0_bare, "l1": l1_bare},
}

results = {
    "seven_no_recovery": [],
    "seven_optimal": [],
    "bgm_no_recovery": [],
    "bgm_optimal": [],
    "gross13_no_recovery": [],
    "gross13_optimal": [],
    "gross17_no_recovery": [],
    "gross17_optimal": [],
    "bare_no_recovery": [],
    "bare_optimal": [],
}

for gamma in gamma_vals:
    print(f"Processing gamma={gamma:.2e}")

    for label, prefix in [
        ("seven-qubit", "seven"),
        ("bgm(3,3,1)", "bgm"),
        ("gross13", "gross13"),
        ("gross17", "gross17"),
        ("bare qubit", "bare"),
    ]:
        spec = codes[label]

        try:
            channel = noisemodel(noise_name, spec["N"], gamma, dt, method)
            fid_no_recovery = no_recovery(spec["rho"], channel)
            fid_opt = optimise(spec["l0"], spec["l1"], channel, solver=solver)

            results[f"{prefix}_no_recovery"].append(abs(1.0 - float(fid_no_recovery)))
            results[f"{prefix}_optimal"].append(abs(1.0 - float(fid_opt)))
        except Exception as exc:
            print(f"  ERROR {label}: {exc}")
            results[f"{prefix}_no_recovery"].append(np.nan)
            results[f"{prefix}_optimal"].append(np.nan)
        finally:
            gc.collect()

for key in results:
    results[key] = np.asarray(results[key], dtype=float)

plt.figure(figsize=(10, 6), dpi=120)

style_map = {
    "seven": {"color": "C0", "marker": "o", "label": "Seven-qubit code"},
    "bgm": {"color": "C1", "marker": "s", "label": "BGM (3,3,1)"},
    "gross13": {"color": "C2", "marker": "D", "label": "Gross 13-qubit"},
    "gross17": {"color": "C3", "marker": "v", "label": "Gross 17-qubit"},
    "bare": {"color": "0.45", "marker": "^", "label": "Bare qubit"},
}

for prefix in ["seven", "bgm", "gross13", "gross17", "bare"]:
    style = style_map[prefix]

    no_rec = results[f"{prefix}_no_recovery"]
    opt_rec = results[f"{prefix}_optimal"]

    mask_no = (no_rec > 0) & np.isfinite(no_rec)
    mask_opt = (opt_rec > 0) & np.isfinite(opt_rec)

    if np.any(mask_no):
        plt.loglog(
            p_vals[mask_no],
            no_rec[mask_no],
            linestyle="--",
            marker="x",
            lw=1.6,
            color=style["color"],
            label=f"{style['label']} no recovery",
        )

    if np.any(mask_opt):
        plt.loglog(
            p_vals[mask_opt],
            opt_rec[mask_opt],
            linestyle="-",
            marker=style["marker"],
            lw=1.8,
            color=style["color"],
            label=f"{style['label']} optimal recovery",
        )

plt.xlabel(r"$p = \amma \elta t$")
plt.ylabel(r"$|1 - F|$")
plt.title("Local symmetric amplitude damping")
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

Processing gamma=1.00e-05


In [ ]:
results

In [ ]:
from pathlib import Path
import csv

if "p_vals" not in globals() or "results" not in globals():
    raise RuntimeError("Run Cell 1 first so p_vals and results are available.")

p_vals_arr = np.asarray(p_vals, dtype=float)
result_keys = list(results.keys())
result_arrays = {k: np.asarray(results[k], dtype=float) for k in result_keys}

for k, arr in result_arrays.items():
    if arr.shape[0] != p_vals_arr.shape[0]:
        raise ValueError(f"Length mismatch: {k} has {arr.shape[0]} points, expected {p_vals_arr.shape[0]}.")

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "README.md").exists():
        repo_root = candidate
        break
if repo_root is None:
    repo_root = cwd

data_dir = repo_root / "datas"
data_dir.mkdir(parents=True, exist_ok=True)
csv_path = data_dir / "local_amp_damp_results.csv"

with csv_path.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["gamma_dt", *result_keys])
    for i in range(p_vals_arr.shape[0]):
        writer.writerow([p_vals_arr[i], *[result_arrays[k][i] for k in result_keys]])

print(f"Saved CSV to: {csv_path}")
print(f"Rows written: {p_vals_arr.shape[0]}")